# Portfolio Trading Performance Report

This notebook reviews the model results from the **portfolio/backtest perspective** first.

Main questions:
- Which models produce the best long-only portfolio returns?
- Which models produce useful long-short portfolios?
- How do those portfolios compare with buy-and-hold over the same evaluation window?
- Are LSTM/GRU/TCN more useful when judged by portfolio metrics than by pure forecast error?

The statistical metrics are still included, but they are supporting evidence. For the management layer, the primary signal is whether the model can rank stocks into portfolios with attractive return, Sharpe, and drawdown behavior.


In [1]:
from __future__ import annotations

from html import escape
from pathlib import Path
import math

import pandas as pd
from IPython.display import HTML, Markdown, display

BASE_DIR = Path.cwd()
REPORT_DIR = BASE_DIR / "outputs" / "reports"
DATA_SPLITS_DIR = BASE_DIR / "data" / "splits"
INITIAL_CAPITAL = 10_000.0
TOP_K_LONGS = 5
BOTTOM_K_SHORTS = 5

HORIZONS = {
    "1d": "1 Day",
    "5d": "5 Day",
}

DEEP_MODELS = ["lstm_default", "gru_default", "tcn_default"]
CORE_TRADING_COLUMNS = [
    "model",
    "long_only_total_return",
    "long_only_sharpe_ratio",
    "long_only_max_drawdown",
    "long_only_excess_vs_buy_hold",
    "long_short_total_return",
    "long_short_sharpe_ratio",
    "long_short_max_drawdown",
    "buy_and_hold_benchmark_total_return",
]

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [2]:
def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run: python src/models/99_gather_report.py")
    return path

model_tables = {
    horizon: pd.read_csv(require_file(REPORT_DIR / f"model_performance_{horizon}.csv"))
    for horizon in HORIZONS
}
stock_tables = {
    horizon: pd.read_csv(require_file(REPORT_DIR / f"stock_model_performance_{horizon}.csv"))
    for horizon in HORIZONS
}

for horizon, df in model_tables.items():
    df["horizon"] = horizon
    df["horizon_label"] = HORIZONS[horizon]
    df["long_only_excess_vs_buy_hold"] = (
        df["long_only_total_return"] - df["buy_and_hold_benchmark_total_return"]
    )
    df["long_only_return_to_drawdown"] = df["long_only_total_return"] / df["long_only_max_drawdown"].abs()
    df["long_short_return_to_drawdown"] = df["long_short_total_return"] / df["long_short_max_drawdown"].abs()

for horizon, df in stock_tables.items():
    df["horizon"] = horizon
    df["horizon_label"] = HORIZONS[horizon]

all_models = pd.concat(model_tables.values(), ignore_index=True)
all_stock_models = pd.concat(stock_tables.values(), ignore_index=True)

print(f"Models evaluated: {sorted(all_models['model'].unique())}")
print(f"Aggregate result rows: {len(all_models)}")
print(f"Stock x model rows: {len(all_stock_models)}")
print(f"Stocks: {all_stock_models['ticker'].nunique()}")


Models evaluated: ['gru_default', 'historical_mean', 'linear_regression', 'lstm_default', 'naive_last_return', 'random_forest', 'ridge_regression', 'tcn_default']
Aggregate result rows: 15
Stock x model rows: 450
Stocks: 30


## Executive Summary: Trading Metrics First

The conclusion changes when we look at portfolio metrics instead of only RMSE:

- The **5-day horizon** is the strongest candidate for the management layer.
- `random_forest` is the best 5-day long-only performer in this run.
- `gru_default` is very strong on 5-day long-only performance and has a positive 5-day long-short result.
- `lstm_default` is much better under long-only portfolio evaluation than its error ranking suggests, though its long-short result is weak.
- `ridge_regression` / `linear_regression` are the cleanest long-short baselines.
- Buy-and-hold remains an important benchmark, not a model. The management layer should always report model excess return versus buy-and-hold.


In [3]:
def pct(value) -> str:
    return "" if pd.isna(value) else f"{value:.2%}"


def number(value, digits=3) -> str:
    return "" if pd.isna(value) else f"{value:.{digits}f}"


def money(value) -> str:
    return "" if pd.isna(value) else f"${value:,.0f}"


def best_row(df: pd.DataFrame, column: str, largest: bool = True) -> pd.Series:
    idx = df[column].idxmax() if largest else df[column].idxmin()
    return df.loc[idx]


def executive_summary() -> pd.DataFrame:
    rows = []
    for horizon, label in HORIZONS.items():
        df = model_tables[horizon]
        for metric, title, fmt, largest in [
            ("long_only_total_return", "Best Long-Only Return", pct, True),
            ("long_only_sharpe_ratio", "Best Long-Only Sharpe", lambda x: number(x, 3), True),
            ("long_only_excess_vs_buy_hold", "Best Excess vs Buy/Hold", pct, True),
            ("long_short_total_return", "Best Long-Short Return", pct, True),
            ("long_short_sharpe_ratio", "Best Long-Short Sharpe", lambda x: number(x, 3), True),
            ("long_only_max_drawdown", "Smallest Long-Only Drawdown", pct, True),
        ]:
            row = best_row(df, metric, largest=largest)
            rows.append({
                "Horizon": label,
                "Question": title,
                "Winner": row["model"],
                "Value": fmt(row[metric]),
            })
    return pd.DataFrame(rows)

summary_df = executive_summary()
display(summary_df)


,Horizon,Question,Winner,Value
0,1 Day,Best Long-Only Return,naive_last_return,31.88%
1,1 Day,Best Long-Only Sharpe,naive_last_return,0.742
2,1 Day,Best Excess vs Buy/Hold,naive_last_return,4.20%
3,1 Day,Best Long-Short Return,ridge_regression,8.49%
4,1 Day,Best Long-Short Sharpe,ridge_regression,0.425
5,1 Day,Smallest Long-Only Drawdown,ridge_regression,-15.61%
6,5 Day,Best Long-Only Return,random_forest,511.67%
7,5 Day,Best Long-Only Sharpe,random_forest,2.346
8,5 Day,Best Excess vs Buy/Hold,random_forest,278.56%
9,5 Day,Best Long-Short Return,ridge_regression,80.60%


## Backtest Scorecards

These tables are sorted by long-only total return because that is the most direct signal for a first portfolio management prototype. The excess column compares each model's long-only return with its own buy-and-hold benchmark window.


In [4]:
def trading_scorecard(df: pd.DataFrame) -> pd.DataFrame:
    out = df[CORE_TRADING_COLUMNS].copy().sort_values("long_only_total_return", ascending=False)
    out = out.rename(columns={
        "model": "Model",
        "long_only_total_return": "Long-Only Return",
        "long_only_sharpe_ratio": "Long-Only Sharpe",
        "long_only_max_drawdown": "Long-Only Max DD",
        "long_only_excess_vs_buy_hold": "Excess vs Buy/Hold",
        "long_short_total_return": "Long-Short Return",
        "long_short_sharpe_ratio": "Long-Short Sharpe",
        "long_short_max_drawdown": "Long-Short Max DD",
        "buy_and_hold_benchmark_total_return": "Buy/Hold Return",
    })
    for col in ["Long-Only Return", "Long-Only Max DD", "Excess vs Buy/Hold", "Long-Short Return", "Long-Short Max DD", "Buy/Hold Return"]:
        out[col] = out[col].map(pct)
    for col in ["Long-Only Sharpe", "Long-Short Sharpe"]:
        out[col] = out[col].map(lambda value: number(value, 3))
    return out

for horizon, label in HORIZONS.items():
    display(Markdown(f"### {label}: Trading Scorecard"))
    display(trading_scorecard(model_tables[horizon]))


### 1 Day: Trading Scorecard

,Model,Long-Only Return,Long-Only Sharpe,Long-Only Max DD,Excess vs Buy/Hold,Long-Short Return,Long-Short Sharpe,Long-Short Max DD,Buy/Hold Return
7,naive_last_return,31.88%,0.742,-21.23%,4.20%,1.73%,0.125,-14.41%,27.68%
3,random_forest,18.69%,0.593,-19.44%,-8.99%,1.49%,0.118,-15.00%,27.68%
4,lstm_default,17.85%,0.507,-20.40%,-7.55%,-1.79%,-0.047,-16.44%,25.40%
0,historical_mean,15.43%,0.498,-19.72%,-12.25%,5.53%,0.318,-15.69%,27.68%
1,ridge_regression,12.57%,0.391,-15.61%,-15.11%,8.49%,0.425,-14.64%,27.68%
2,linear_regression,12.57%,0.391,-15.61%,-15.11%,8.49%,0.425,-14.64%,27.68%
5,gru_default,6.35%,0.245,-21.61%,-19.05%,-16.84%,-0.860,-19.61%,25.40%
6,tcn_default,-0.64%,0.073,-24.62%,-26.04%,-1.82%,-0.052,-15.65%,25.40%


### 5 Day: Trading Scorecard

,Model,Long-Only Return,Long-Only Sharpe,Long-Only Max DD,Excess vs Buy/Hold,Long-Short Return,Long-Short Sharpe,Long-Short Max DD,Buy/Hold Return
3,random_forest,511.67%,2.346,-46.33%,278.56%,48.63%,0.953,-31.32%,233.11%
4,gru_default,353.75%,2.188,-56.26%,168.41%,52.57%,1.110,-33.16%,185.35%
1,ridge_regression,237.72%,1.574,-49.06%,4.61%,80.60%,1.259,-36.15%,233.11%
2,linear_regression,237.72%,1.574,-49.06%,4.61%,80.60%,1.259,-36.15%,233.11%
5,lstm_default,209.34%,1.568,-51.06%,24.00%,-17.18%,-0.288,-45.94%,185.35%
6,tcn_default,147.67%,1.308,-45.32%,-37.67%,39.44%,0.864,-31.55%,185.35%
0,historical_mean,96.58%,1.069,-62.59%,-136.53%,26.41%,0.610,-55.76%,233.11%


## Visualization Helpers

The project environment does not currently include plotting libraries, so these charts are rendered as self-contained HTML/SVG. They run anywhere the notebook runs.


In [5]:
PALETTE = {
    "long_only": "#2f855a",
    "long_short": "#2b6cb0",
    "buy_hold": "#805ad5",
    "positive": "#2f855a",
    "negative": "#c53030",
    "neutral": "#718096",
    "deep": "#dd6b20",
    "baseline": "#2b6cb0",
}


def model_family(model: str) -> str:
    return "deep" if model in DEEP_MODELS else "baseline"


def chart_css() -> str:
    return """
    <style>
      .note { color: #4a5568; font-size: 12px; margin: 4px 0 10px 0; }
      .metric-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(210px, 1fr)); gap: 12px; margin: 12px 0 22px 0; }
      .metric-card { border: 1px solid #e2e8f0; border-radius: 6px; padding: 12px; background: #f8fafc; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; }
      .metric-title { color: #4a5568; font-size: 12px; margin-bottom: 6px; }
      .metric-value { font-size: 22px; font-weight: 700; color: #1a202c; }
      .metric-sub { color: #4a5568; font-size: 12px; margin-top: 4px; }
      .bar-row { display: grid; grid-template-columns: 175px 1fr 95px; gap: 10px; align-items: center; margin: 5px 0; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; font-size: 13px; }
      .bar-label { white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
      .bar-track { height: 19px; position: relative; background: #edf2f7; border-radius: 3px; overflow: hidden; }
      .bar { position: absolute; top: 0; bottom: 0; border-radius: 3px; }
      .zero { position: absolute; top: 0; bottom: 0; border-left: 1px solid #2d3748; opacity: 0.5; }
      .bar-value { text-align: right; font-variant-numeric: tabular-nums; }
      .chart-title { font-weight: 700; margin: 16px 0 6px 0; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; }
      .legend { display:flex; flex-wrap:wrap; gap:12px; font-size:12px; color:#4a5568; margin:6px 0 10px 0; }
      .legend-item { display:inline-flex; align-items:center; gap:5px; }
      .swatch { width:11px; height:11px; border-radius:2px; display:inline-block; }
    </style>
    """


def metric_cards(df: pd.DataFrame, horizon_label: str) -> str:
    best_lo = best_row(df, "long_only_total_return")
    best_ls = best_row(df, "long_short_total_return")
    best_excess = best_row(df, "long_only_excess_vs_buy_hold")
    best_sharpe = best_row(df, "long_only_sharpe_ratio")
    return chart_css() + f"""
    <div class="metric-grid">
      <div class="metric-card"><div class="metric-title">{escape(horizon_label)} best long-only return</div><div class="metric-value">{pct(best_lo['long_only_total_return'])}</div><div class="metric-sub">{escape(best_lo['model'])}, Sharpe {number(best_lo['long_only_sharpe_ratio'], 2)}</div></div>
      <div class="metric-card"><div class="metric-title">{escape(horizon_label)} best long-short return</div><div class="metric-value">{pct(best_ls['long_short_total_return'])}</div><div class="metric-sub">{escape(best_ls['model'])}, Sharpe {number(best_ls['long_short_sharpe_ratio'], 2)}</div></div>
      <div class="metric-card"><div class="metric-title">{escape(horizon_label)} best excess vs buy/hold</div><div class="metric-value">{pct(best_excess['long_only_excess_vs_buy_hold'])}</div><div class="metric-sub">{escape(best_excess['model'])}</div></div>
      <div class="metric-card"><div class="metric-title">{escape(horizon_label)} best long-only Sharpe</div><div class="metric-value">{number(best_sharpe['long_only_sharpe_ratio'], 2)}</div><div class="metric-sub">{escape(best_sharpe['model'])}, return {pct(best_sharpe['long_only_total_return'])}</div></div>
    </div>
    """


def horizontal_bar_chart(
    df: pd.DataFrame,
    label_col: str,
    value_col: str,
    title: str,
    formatter=pct,
    ascending: bool = False,
    color_col: str | None = None,
    max_rows: int | None = None,
) -> str:
    chart_df = df[[label_col, value_col] + ([color_col] if color_col else [])].dropna().copy()
    chart_df = chart_df.sort_values(value_col, ascending=ascending)
    if max_rows:
        chart_df = chart_df.head(max_rows)
    if chart_df.empty:
        return chart_css() + f"<div class='chart-title'>{escape(title)}</div><p>No data.</p>"

    low = min(0.0, float(chart_df[value_col].min()))
    high = max(0.0, float(chart_df[value_col].max()))
    span = high - low if not math.isclose(high, low) else 1.0
    zero = (-low / span) * 100
    rows = []

    for _, row in chart_df.iterrows():
        label = escape(str(row[label_col]))
        value = float(row[value_col])
        width = abs(value) / span * 100
        left = zero if value >= 0 else zero - width
        if color_col:
            color = row[color_col]
        else:
            color = PALETTE["positive"] if value >= 0 else PALETTE["negative"]
        rows.append(f"""
        <div class="bar-row">
          <div class="bar-label">{label}</div>
          <div class="bar-track"><div class="zero" style="left:{zero:.2f}%"></div><div class="bar" style="left:{left:.2f}%; width:{width:.2f}%; background:{color};"></div></div>
          <div class="bar-value">{escape(formatter(value))}</div>
        </div>
        """)

    return chart_css() + f"<div class='chart-title'>{escape(title)}</div>{''.join(rows)}"


def grouped_return_svg(df: pd.DataFrame, title: str, width: int = 980, height: int = 420) -> str:
    plot_df = df.sort_values("long_only_total_return", ascending=False).reset_index(drop=True)
    metrics = [
        ("long_only_total_return", "Long Only", PALETTE["long_only"]),
        ("long_short_total_return", "Long-Short", PALETTE["long_short"]),
        ("buy_and_hold_benchmark_total_return", "Buy/Hold", PALETTE["buy_hold"]),
    ]
    values = [float(v) for col, _, _ in metrics for v in plot_df[col].dropna()]
    low = min(0.0, min(values))
    high = max(0.0, max(values))
    if math.isclose(low, high):
        high = low + 1
    pad_l, pad_r, pad_t, pad_b = 70, 20, 44, 95
    inner_w = width - pad_l - pad_r
    inner_h = height - pad_t - pad_b
    group_w = inner_w / max(1, len(plot_df))
    bar_w = min(18, group_w / 4)

    def y(value):
        return pad_t + inner_h - ((value - low) / (high - low)) * inner_h

    parts = [
        f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' xmlns='http://www.w3.org/2000/svg'>",
        f"<text x='{pad_l}' y='20' font-size='16' font-weight='700'>{escape(title)}</text>",
    ]
    for tick in [0, 0.25, 0.5, 0.75, 1.0]:
        val = low + (high - low) * tick
        yy = y(val)
        parts.append(f"<line x1='{pad_l}' y1='{yy:.1f}' x2='{pad_l + inner_w}' y2='{yy:.1f}' stroke='#e2e8f0'/>")
        parts.append(f"<text x='{pad_l - 8}' y='{yy + 4:.1f}' font-size='11' text-anchor='end'>{val:.0%}</text>")
    zero_y = y(0)
    parts.append(f"<line x1='{pad_l}' y1='{zero_y:.1f}' x2='{pad_l + inner_w}' y2='{zero_y:.1f}' stroke='#1a202c'/>")

    for i, (_, row) in enumerate(plot_df.iterrows()):
        group_x = pad_l + i * group_w + group_w / 2
        for j, (col, _, color) in enumerate(metrics):
            value = float(row[col])
            x = group_x + (j - 1) * (bar_w + 3) - bar_w / 2
            top = min(y(value), zero_y)
            bar_h = abs(y(value) - zero_y)
            parts.append(f"<rect x='{x:.1f}' y='{top:.1f}' width='{bar_w:.1f}' height='{bar_h:.1f}' fill='{color}' rx='2'/>")
        parts.append(f"<text x='{group_x:.1f}' y='{height - 48}' font-size='10' text-anchor='end' transform='rotate(-45 {group_x:.1f},{height - 48})'>{escape(str(row['model']))}</text>")

    legend_x = pad_l
    for _, label, color in metrics:
        parts.append(f"<rect x='{legend_x}' y='{height - 24}' width='11' height='11' fill='{color}' rx='2'/>")
        parts.append(f"<text x='{legend_x + 16}' y='{height - 14}' font-size='12'>{escape(label)}</text>")
        legend_x += 125
    parts.append("</svg>")
    return ''.join(parts)


def risk_return_svg(df: pd.DataFrame, book: str, title: str, width: int = 920, height: int = 420) -> str:
    ret_col = f"{book}_total_return"
    sharpe_col = f"{book}_sharpe_ratio"
    dd_col = f"{book}_max_drawdown"
    plot_df = df[["model", ret_col, sharpe_col, dd_col]].dropna().copy()
    plot_df["drawdown_abs"] = plot_df[dd_col].abs()
    if plot_df.empty:
        return f"<p>No data for {escape(book)}</p>"

    x_min, x_max = 0.0, float(plot_df["drawdown_abs"].max()) * 1.08
    y_min = min(0.0, float(plot_df[ret_col].min()))
    y_max = float(plot_df[ret_col].max()) * 1.10
    if math.isclose(y_min, y_max):
        y_max = y_min + 1
    pad_l, pad_r, pad_t, pad_b = 75, 170, 42, 55
    inner_w = width - pad_l - pad_r
    inner_h = height - pad_t - pad_b

    def sx(value):
        return pad_l + (value - x_min) / (x_max - x_min) * inner_w

    def sy(value):
        return pad_t + inner_h - (value - y_min) / (y_max - y_min) * inner_h

    parts = [
        f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' xmlns='http://www.w3.org/2000/svg'>",
        f"<text x='{pad_l}' y='20' font-size='16' font-weight='700'>{escape(title)}</text>",
        f"<line x1='{pad_l}' y1='{pad_t + inner_h}' x2='{pad_l + inner_w}' y2='{pad_t + inner_h}' stroke='#2d3748'/>",
        f"<line x1='{pad_l}' y1='{pad_t}' x2='{pad_l}' y2='{pad_t + inner_h}' stroke='#2d3748'/>",
        f"<text x='{pad_l + inner_w/2}' y='{height - 10}' font-size='12' text-anchor='middle'>Max drawdown magnitude</text>",
        f"<text x='14' y='{pad_t + inner_h/2}' font-size='12' transform='rotate(-90 14,{pad_t + inner_h/2})' text-anchor='middle'>Total return</text>",
    ]

    for tick in [0, 0.25, 0.5, 0.75, 1.0]:
        xv = x_min + (x_max - x_min) * tick
        yv = y_min + (y_max - y_min) * tick
        x = sx(xv)
        y = sy(yv)
        parts.append(f"<line x1='{x:.1f}' y1='{pad_t}' x2='{x:.1f}' y2='{pad_t + inner_h}' stroke='#edf2f7'/>")
        parts.append(f"<text x='{x:.1f}' y='{pad_t + inner_h + 17}' font-size='10' text-anchor='middle'>{xv:.0%}</text>")
        parts.append(f"<line x1='{pad_l}' y1='{y:.1f}' x2='{pad_l + inner_w}' y2='{y:.1f}' stroke='#edf2f7'/>")
        parts.append(f"<text x='{pad_l - 8}' y='{y + 4:.1f}' font-size='10' text-anchor='end'>{yv:.0%}</text>")

    if y_min < 0 < y_max:
        parts.append(f"<line x1='{pad_l}' y1='{sy(0):.1f}' x2='{pad_l + inner_w}' y2='{sy(0):.1f}' stroke='#718096' stroke-dasharray='4 3'/>")

    for _, row in plot_df.iterrows():
        x = sx(float(row["drawdown_abs"]))
        y = sy(float(row[ret_col]))
        family = model_family(row["model"])
        color = PALETTE[family]
        radius = 5 + max(0, min(8, float(row[sharpe_col]) * 2)) if not pd.isna(row[sharpe_col]) else 6
        parts.append(f"<circle cx='{x:.1f}' cy='{y:.1f}' r='{radius:.1f}' fill='{color}' fill-opacity='0.82' stroke='#1a202c' stroke-width='0.8'/>")
        parts.append(f"<text x='{x + 8:.1f}' y='{y + 4:.1f}' font-size='11'>{escape(str(row['model']))}</text>")

    parts.append(f"<rect x='{width - 150}' y='35' width='11' height='11' fill='{PALETTE['baseline']}' rx='2'/><text x='{width - 133}' y='45' font-size='12'>Baseline</text>")
    parts.append(f"<rect x='{width - 150}' y='55' width='11' height='11' fill='{PALETTE['deep']}' rx='2'/><text x='{width - 133}' y='65' font-size='12'>Deep model</text>")
    parts.append("</svg>")
    return ''.join(parts)


def color_for_value(value: float, low: float, high: float, lower_is_better: bool = False) -> str:
    if pd.isna(value):
        return "#f3f4f6"
    ratio = 0.5 if math.isclose(high, low) else (float(value) - low) / (high - low)
    if lower_is_better:
        ratio = 1 - ratio
    hue = 6 + (120 - 6) * max(0, min(1, ratio))
    return f"hsl({hue:.0f}, 70%, 84%)"


def heatmap_html(df: pd.DataFrame, title: str, value_col: str = "rmse", lower_is_better: bool = True) -> str:
    pivot = df.pivot_table(index="ticker", columns="model", values=value_col, aggfunc="mean")
    if pivot.empty:
        return f"<h4>{escape(title)}</h4><p>No data.</p>"
    model_order = pivot.mean(axis=0).sort_values(ascending=lower_is_better).index.tolist()
    pivot = pivot[model_order]
    low, high = float(pivot.min().min()), float(pivot.max().max())
    header = ''.join(f"<th>{escape(str(col))}</th>" for col in pivot.columns)
    body_rows = []
    for ticker, row in pivot.iterrows():
        best_value = row.min() if lower_is_better else row.max()
        cells = []
        for _, value in row.items():
            bg = color_for_value(value, low, high, lower_is_better)
            border = "2px solid #1a202c" if not pd.isna(value) and math.isclose(float(value), float(best_value), rel_tol=1e-12, abs_tol=1e-12) else "1px solid #e2e8f0"
            shown = "" if pd.isna(value) else f"{value:.4f}"
            cells.append(f"<td style='background:{bg}; border:{border};'>{shown}</td>")
        body_rows.append(f"<tr><th>{escape(str(ticker))}</th>{''.join(cells)}</tr>")
    return chart_css() + f"""
    <h4>{escape(title)}</h4>
    <div class="note">Lower RMSE is better. The outlined cell is the best model for that ticker.</div>
    <div style="overflow-x:auto;"><table style="border-collapse:collapse;font-size:11px;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;">
      <thead><tr><th style="border:1px solid #e2e8f0;padding:4px 6px;background:#f8fafc;">Ticker</th>{header}</tr></thead>
      <tbody>{''.join(body_rows)}</tbody>
    </table></div>
    """


## Total Return: Long-Only vs Long-Short vs Buy-and-Hold

This is the key view. It directly answers whether model rankings are useful as portfolio inputs.


In [6]:
for horizon, label in HORIZONS.items():
    display(Markdown(f"### {label}: Portfolio Return Summary"))
    display(HTML(metric_cards(model_tables[horizon], label)))
    display(HTML(grouped_return_svg(model_tables[horizon], f"{label}: Total Return by Portfolio Type")))


### 1 Day: Portfolio Return Summary

### 5 Day: Portfolio Return Summary

## Excess Return Versus Buy-and-Hold

A model can look strong in absolute terms during a rising market. Excess return asks whether its stock ranking added value above a simple equal-weight buy-and-hold benchmark over the same prediction window.


In [7]:
for horizon, label in HORIZONS.items():
    df = model_tables[horizon].copy()
    df["family_color"] = df["model"].map(lambda model: PALETTE["deep"] if model in DEEP_MODELS else PALETTE["baseline"])
    display(Markdown(f"### {label}: Long-Only Excess Return vs Buy/Hold"))
    display(HTML(horizontal_bar_chart(
        df,
        "model",
        "long_only_excess_vs_buy_hold",
        f"{label}: Long-Only Excess Return",
        formatter=pct,
        ascending=False,
        color_col="family_color",
    ) + "<div class='legend'><span class='legend-item'><span class='swatch' style='background:#2b6cb0'></span>Baseline</span><span class='legend-item'><span class='swatch' style='background:#dd6b20'></span>Deep model</span></div>"))


### 1 Day: Long-Only Excess Return vs Buy/Hold

### 5 Day: Long-Only Excess Return vs Buy/Hold

## Sharpe And Drawdown

Return without risk context can be misleading. These views show whether the return came with acceptable volatility and drawdown.


In [8]:
for horizon, label in HORIZONS.items():
    display(Markdown(f"### {label}: Long-Only Sharpe"))
    display(HTML(horizontal_bar_chart(
        model_tables[horizon],
        "model",
        "long_only_sharpe_ratio",
        f"{label}: Long-Only Sharpe Ratio",
        formatter=lambda value: number(value, 2),
        ascending=False,
    )))
    display(Markdown(f"### {label}: Long-Short Sharpe"))
    display(HTML(horizontal_bar_chart(
        model_tables[horizon],
        "model",
        "long_short_sharpe_ratio",
        f"{label}: Long-Short Sharpe Ratio",
        formatter=lambda value: number(value, 2),
        ascending=False,
    )))
    dd_df = model_tables[horizon].copy()
    dd_df["long_only_drawdown_abs"] = dd_df["long_only_max_drawdown"].abs()
    display(Markdown(f"### {label}: Long-Only Max Drawdown Magnitude"))
    display(HTML(horizontal_bar_chart(
        dd_df,
        "model",
        "long_only_drawdown_abs",
        f"{label}: Smaller Drawdown Is Better",
        formatter=pct,
        ascending=True,
    )))


### 1 Day: Long-Only Sharpe

### 1 Day: Long-Short Sharpe

### 1 Day: Long-Only Max Drawdown Magnitude

### 5 Day: Long-Only Sharpe

### 5 Day: Long-Short Sharpe

### 5 Day: Long-Only Max Drawdown Magnitude

## Risk/Return Maps

These charts put return and drawdown on the same page. Points higher and farther left are better. Orange points are LSTM/GRU/TCN; blue points are baselines.


In [9]:
for horizon, label in HORIZONS.items():
    display(HTML(risk_return_svg(model_tables[horizon], "long_only", f"{label}: Long-Only Risk / Return")))
    display(HTML(risk_return_svg(model_tables[horizon], "long_short", f"{label}: Long-Short Risk / Return")))


## Deep Model Portfolio Lens

The deep models are not the best by forecast error, but portfolio metrics tell a more interesting story. This table isolates LSTM/GRU/TCN and compares them with their own buy-and-hold benchmark windows.


In [10]:
deep_rows = all_models[all_models["model"].isin(DEEP_MODELS)].copy()
deep_view = deep_rows[[
    "horizon_label",
    "model",
    "long_only_total_return",
    "buy_and_hold_benchmark_total_return",
    "long_only_excess_vs_buy_hold",
    "long_only_sharpe_ratio",
    "long_short_total_return",
    "long_short_sharpe_ratio",
    "long_only_max_drawdown",
]].sort_values(["horizon_label", "long_only_total_return"], ascending=[True, False])

deep_display = deep_view.rename(columns={
    "horizon_label": "Horizon",
    "model": "Model",
    "long_only_total_return": "Long-Only Return",
    "buy_and_hold_benchmark_total_return": "Buy/Hold Return",
    "long_only_excess_vs_buy_hold": "Excess vs Buy/Hold",
    "long_only_sharpe_ratio": "Long-Only Sharpe",
    "long_short_total_return": "Long-Short Return",
    "long_short_sharpe_ratio": "Long-Short Sharpe",
    "long_only_max_drawdown": "Long-Only Max DD",
})
for col in ["Long-Only Return", "Buy/Hold Return", "Excess vs Buy/Hold", "Long-Short Return", "Long-Only Max DD"]:
    deep_display[col] = deep_display[col].map(pct)
for col in ["Long-Only Sharpe", "Long-Short Sharpe"]:
    deep_display[col] = deep_display[col].map(lambda value: number(value, 3))

display(deep_display)

for horizon, label in HORIZONS.items():
    df = model_tables[horizon][model_tables[horizon]["model"].isin(DEEP_MODELS)].copy()
    display(HTML(grouped_return_svg(df, f"{label}: Deep Models - Long-Only, Long-Short, Buy/Hold", width=760, height=360)))


,Horizon,Model,Long-Only Return,Buy/Hold Return,Excess vs Buy/Hold,Long-Only Sharpe,Long-Short Return,Long-Short Sharpe,Long-Only Max DD
4,1 Day,lstm_default,17.85%,25.40%,-7.55%,0.507,-1.79%,-0.047,-20.40%
5,1 Day,gru_default,6.35%,25.40%,-19.05%,0.245,-16.84%,-0.860,-21.61%
6,1 Day,tcn_default,-0.64%,25.40%,-26.04%,0.073,-1.82%,-0.052,-24.62%
12,5 Day,gru_default,353.75%,185.35%,168.41%,2.188,52.57%,1.110,-56.26%
13,5 Day,lstm_default,209.34%,185.35%,24.00%,1.568,-17.18%,-0.288,-51.06%
14,5 Day,tcn_default,147.67%,185.35%,-37.67%,1.308,39.44%,0.864,-45.32%


## Equity Curves From Saved Predictions

These curves recreate the top/bottom ranking logic from the saved prediction files. They help show whether returns were steady or dominated by a few jumps.


In [11]:
def prediction_path(model_row: pd.Series, horizon: str) -> Path:
    return BASE_DIR / str(model_row["model_dir"]) / f"predictions_{horizon}_test.csv"


def load_test_predictions(model: str, horizon: str) -> pd.DataFrame:
    rows = model_tables[horizon][model_tables[horizon]["model"] == model]
    if rows.empty:
        raise ValueError(f"Model {model!r} not found for horizon {horizon!r}")
    path = prediction_path(rows.iloc[0], horizon)
    df = pd.read_csv(require_file(path), parse_dates=["Date"])
    df["model"] = model
    return df


def strategy_returns(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for date, group in pred_df.groupby("Date"):
        if len(group) < TOP_K_LONGS + BOTTOM_K_SHORTS:
            continue
        ordered = group.sort_values("y_pred", ascending=False)
        top = ordered.head(TOP_K_LONGS)["y_true"].mean()
        bottom = ordered.tail(BOTTOM_K_SHORTS)["y_true"].mean()
        rows.append({
            "Date": date,
            "long_only": math.exp(top) - 1,
            "long_short": math.exp((top - bottom) / 2.0) - 1,
            "buy_hold": math.exp(group["y_true"].mean()) - 1,
        })
    return pd.DataFrame(rows).sort_values("Date")


def equity_curve(pred_df: pd.DataFrame, book: str = "long_only") -> pd.Series:
    returns = strategy_returns(pred_df)
    equity = INITIAL_CAPITAL * (1 + returns[book]).cumprod()
    equity.index = returns["Date"]
    return equity


def line_chart_svg(series_map: dict[str, pd.Series], title: str, width: int = 960, height: int = 390) -> str:
    clean = {name: s.dropna().sort_index() for name, s in series_map.items() if not s.dropna().empty}
    if not clean:
        return f"<h4>{escape(title)}</h4><p>No series to plot.</p>"

    all_values = pd.concat(clean.values())
    y_min = min(INITIAL_CAPITAL * 0.8, float(all_values.min()))
    y_max = float(all_values.max()) * 1.08
    if math.isclose(y_min, y_max):
        y_max = y_min + 1
    pad_l, pad_r, pad_t, pad_b = 75, 150, 34, 44
    inner_w = width - pad_l - pad_r
    inner_h = height - pad_t - pad_b
    colors = ["#2f855a", "#2b6cb0", "#805ad5", "#dd6b20", "#c53030", "#319795", "#718096", "#d69e2e"]

    def sy(value):
        return pad_t + inner_h - ((float(value) - y_min) / (y_max - y_min)) * inner_h

    parts = [
        f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' xmlns='http://www.w3.org/2000/svg'>",
        f"<text x='{pad_l}' y='20' font-size='16' font-weight='700'>{escape(title)}</text>",
        f"<line x1='{pad_l}' y1='{pad_t + inner_h}' x2='{pad_l + inner_w}' y2='{pad_t + inner_h}' stroke='#2d3748'/>",
        f"<line x1='{pad_l}' y1='{pad_t}' x2='{pad_l}' y2='{pad_t + inner_h}' stroke='#2d3748'/>",
    ]
    for tick in [0, 0.25, 0.5, 0.75, 1.0]:
        val = y_min + (y_max - y_min) * tick
        y = sy(val)
        parts.append(f"<line x1='{pad_l}' y1='{y:.1f}' x2='{pad_l + inner_w}' y2='{y:.1f}' stroke='#e2e8f0'/>")
        parts.append(f"<text x='{pad_l - 8}' y='{y + 4:.1f}' font-size='10' text-anchor='end'>${val:,.0f}</text>")

    for idx, (name, series) in enumerate(clean.items()):
        points = []
        n = len(series)
        for i, value in enumerate(series.values):
            x = pad_l + (i / max(1, n - 1)) * inner_w
            y = sy(value)
            points.append(f"{x:.1f},{y:.1f}")
        color = colors[idx % len(colors)]
        parts.append(f"<polyline points='{' '.join(points)}' fill='none' stroke='{color}' stroke-width='2'/>")
        end_x, end_y = points[-1].split(',')
        parts.append(f"<text x='{float(end_x) + 8:.1f}' y='{float(end_y) + 4:.1f}' font-size='11' fill='{color}'>{escape(name)}</text>")

    first_date = min(s.index.min() for s in clean.values()).date()
    last_date = max(s.index.max() for s in clean.values()).date()
    parts.append(f"<text x='{pad_l}' y='{height - 10}' font-size='11'>{first_date}</text>")
    parts.append(f"<text x='{pad_l + inner_w}' y='{height - 10}' font-size='11' text-anchor='end'>{last_date}</text>")
    parts.append("</svg>")
    return ''.join(parts)


In [12]:
selected_long_only = {
    "1d": ["naive_last_return", "random_forest", "lstm_default", "historical_mean"],
    "5d": ["random_forest", "gru_default", "lstm_default", "ridge_regression", "tcn_default"],
}
selected_long_short = {
    "1d": ["ridge_regression", "linear_regression", "historical_mean", "lstm_default"],
    "5d": ["ridge_regression", "linear_regression", "gru_default", "random_forest", "tcn_default"],
}

for horizon, label in HORIZONS.items():
    available = set(model_tables[horizon]["model"])
    lo_curves = {}
    for model in selected_long_only[horizon]:
        if model in available:
            preds = load_test_predictions(model, horizon)
            lo_curves[f"{model} LO"] = equity_curve(preds, "long_only")
            if model == selected_long_only[horizon][0]:
                lo_curves[f"{model} BH"] = equity_curve(preds, "buy_hold")
    display(HTML(line_chart_svg(lo_curves, f"{label}: Long-Only Equity Curves vs Benchmark")))

    ls_curves = {}
    for model in selected_long_short[horizon]:
        if model in available:
            preds = load_test_predictions(model, horizon)
            ls_curves[f"{model} LS"] = equity_curve(preds, "long_short")
    display(HTML(line_chart_svg(ls_curves, f"{label}: Long-Short Equity Curves")))


## Stock-Level Context

The management layer will allocate by ticker, so aggregate performance is not enough. This heatmap keeps the stock-level forecast consistency visible while leaving portfolio performance as the main decision metric.


In [13]:
for horizon, label in HORIZONS.items():
    display(HTML(heatmap_html(stock_tables[horizon], f"{label}: Stock x Model RMSE Context")))


Ticker,historical_mean,ridge_regression,linear_regression,random_forest,lstm_default,gru_default,tcn_default,naive_last_return
AAPL,0.0172,0.0173,0.0173,0.0180,0.0178,0.0181,0.0187,0.0237
ABT,0.0135,0.0136,0.0136,0.0135,0.0139,0.0140,0.0145,0.0194
ADBE,0.0220,0.0221,0.0221,0.0222,0.0226,0.0231,0.0247,0.0316
AMZN,0.0197,0.0196,0.0196,0.0199,0.0200,0.0204,0.0204,0.0279
BA,0.0223,0.0223,0.0223,0.0226,0.0221,0.0231,0.0230,0.0311
BAC,0.0158,0.0158,0.0158,0.0157,0.0162,0.0170,0.0170,0.0214
CAT,0.0188,0.0190,0.0190,0.0194,0.0193,0.0200,0.0201,0.0266
COST,0.0128,0.0130,0.0130,0.0134,0.0141,0.0133,0.0162,0.0178
CRM,0.0225,0.0225,0.0225,0.0226,0.0233,0.0234,0.0238,0.0327
CVX,0.0141,0.0139,0.0139,0.0138,0.0146,0.0154,0.0154,0.0195


Ticker,historical_mean,ridge_regression,linear_regression,random_forest,gru_default,lstm_default,tcn_default
AAPL,0.0403,0.0404,0.0404,0.0404,0.0425,0.0434,0.0428
ABT,0.0289,0.0289,0.0289,0.0295,0.0301,0.0301,0.0307
ADBE,0.0492,0.0489,0.0489,0.0497,0.0500,0.0512,0.0511
AMZN,0.0422,0.0421,0.0421,0.0422,0.0428,0.0423,0.0457
BA,0.0516,0.0512,0.0512,0.0512,0.0507,0.0594,0.0565
BAC,0.0362,0.0362,0.0362,0.0364,0.0371,0.0414,0.0404
CAT,0.0418,0.0430,0.0430,0.0427,0.0432,0.0536,0.0554
COST,0.0286,0.0288,0.0288,0.0292,0.0342,0.0321,0.0346
CRM,0.0485,0.0485,0.0485,0.0484,0.0480,0.0505,0.0513
CVX,0.0324,0.0326,0.0326,0.0335,0.0339,0.0360,0.0358


## Supporting Forecast Metrics

Forecast error still matters because unstable or noisy forecasts can produce fragile portfolio rankings. This section is intentionally secondary: it is here to keep us honest, not to override the portfolio evidence.


In [14]:
def forecast_scorecard(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["model", "test_mae", "test_rmse", "test_directional_accuracy"]
    out = df[cols].copy().sort_values("test_rmse")
    out = out.rename(columns={
        "model": "Model",
        "test_mae": "Test MAE",
        "test_rmse": "Test RMSE",
        "test_directional_accuracy": "Directional Accuracy",
    })
    out["Test MAE"] = out["Test MAE"].map(lambda value: number(value, 6))
    out["Test RMSE"] = out["Test RMSE"].map(lambda value: number(value, 6))
    out["Directional Accuracy"] = out["Directional Accuracy"].map(pct)
    return out

for horizon, label in HORIZONS.items():
    display(Markdown(f"### {label}: Forecast Metrics"))
    display(forecast_scorecard(model_tables[horizon]))


### 1 Day: Forecast Metrics

,Model,Test MAE,Test RMSE,Directional Accuracy
0,historical_mean,0.011715,0.017563,52.60%
1,ridge_regression,0.011803,0.017633,50.76%
2,linear_regression,0.011803,0.017633,50.76%
3,random_forest,0.011772,0.017814,52.57%
4,lstm_default,0.012063,0.018031,50.79%
5,gru_default,0.012136,0.018331,51.02%
6,tcn_default,0.012788,0.018851,48.86%
7,naive_last_return,0.017056,0.025033,50.29%


### 5 Day: Forecast Metrics

,Model,Test MAE,Test RMSE,Directional Accuracy
0,historical_mean,0.027986,0.038875,54.55%
1,ridge_regression,0.028136,0.038930,52.66%
2,linear_regression,0.028136,0.038930,52.66%
3,random_forest,0.028177,0.039180,54.38%
4,gru_default,0.029155,0.039897,50.61%
5,lstm_default,0.030482,0.042148,48.28%
6,tcn_default,0.031016,0.042335,48.61%


## Final Summary And Management Recommendation

Portfolio metric summary:

- **1-day horizon:** positive, but not the best foundation for management. The best long-only result is `naive_last_return`, and `lstm_default` is also positive, but buy-and-hold is hard to beat and long-short results are not strong.
- **5-day horizon:** this is the promising management horizon. `random_forest` produces the strongest long-only result, while `ridge_regression` / `linear_regression` produce the strongest long-short profile.
- **Deep models:** viewed through portfolio metrics, the deep models are more useful than the error tables suggest. `gru_default` is especially strong on 5-day long-only return and Sharpe. `lstm_default` beats its comparable buy-and-hold window on 5-day long-only return, though its long-short result is weak. `tcn_default` has positive 5-day long-short behavior but weaker long-only excess.
- **Buy-and-hold:** it should stay as a required benchmark in every management report. It is not enough for a model to make money; the ranking needs to add value over passive exposure.

Recommendation:

Yes, we can move to the management part. The first management version should be built as a **5-day portfolio construction/backtesting layer**, with `random_forest`, `gru_default`, and `ridge_regression`/`linear_regression` as the first candidate signals. Add costs, turnover, holding-period accounting, position limits, and drawdown controls before treating the results as production-grade.
